In [ ]:
## Data Access using App
spark.conf.set("fs.azure.account.auth.type.datalakestorage001123.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakestorage001123.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakestorage001123.dfs.core.windows.net", "8011e294-5740-40bc-ba2d-7bd9fe98d081")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakestorage001123.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakestorage001123.dfs.core.windows.net", "https://login.core.windows.net/b562c482-1c67-4cda-8c85-1c1f846c6054/oauth2/token")



In [ ]:
df_calendar = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Calendar")

df_customer = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Customers")

df_procat = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Product_Categories")

df_product = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Products")

df_return = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Returns")

# It will loaded all the sales folder with "Sales_" into the dataframe
df_sales = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Sales*")

df_ter = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Territories")

df_subcat = spark.read.format("csv").option("header",True).option("inferSchema", True).load("abfss://bronze@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Subcategories")

In [ ]:
### import these files to use these functions
from pyspark.sql.functions import *
form pyspark.sql.types import *


df_calendar.display()

## withColumn: to add a new column
df_calendar.withColumn("Month", month(col("Date")))\
           .withColumn("Year", year(col("Date")))

df_calendar.display()

## some people save in parquet format, some people saves in delta format. 
# Parquet as very efficient for analytics, 
# delta build on top of parquet is good for reliable data, can do incremental loads, multiple users querying at the same time 
# mode: append (data v1, v2), overwrite (v2 overwrite v1), ignore (will not overwrite v1 nor generate error to tell you), error(generate error to tell you)
df_calendar.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()



df_customer.display()
df_customer = df_customer.withColumn("fullName", concat(col("Prefix"), lit(" "), col("FirstName"), lit(" "), col("LastName")))
df_customer.display()
df_customer.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Customers")\
    .save()


df_subcat.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Subcategories")\
    .save()



## This is product SKU, but I just want to fetch the first 2 letters
df_product.display()
### We only want to transform the data, not adding new column
df_product = df_product.withColumn("ProductSKU", split(col("ProductSKU"), ",")[0])\
                        .withColumn("ProductName", split(col("ProductName"), " ")[0])

df_product.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Products")\
    .save()

df_return.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()

df_sales.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Sales")\
    .save()

df_ter.write.format("parquet").mode("append")\
    .optipn("path", "abfss://silver@datalakestorage001123.dfs.core.windows.net/AdventureWorks_Territories")\
    .save()


In [ ]:
#### Now, let's cover how we can do aggregated data
df_sales = df_sales.withColumn("StockDate", to_timestamp("StockDate"))
df_sales = df_sales.withColumn("OrderNumber", regexp_replace(col("OrderNumber"), "S", "T"))
df_sales = df_sales.withColumn("multiply", col("OrderLineItem") * col("OrderQuantity"))
df_sales.display()

### Sales Analysis
- how many order sales we made in every day

In [ ]:
df_sales.groupBy("OrderDate").agg(count("OrderNumber").alias("total_order"))


# Show the visualization function, where we can see line chart


In [ ]:
## how many countries in each region
df_ter.display()

# x-axis - country, y-axis = region


### Do a bit of extension from here